# FlowSlider

In [1]:
import sys
import os
import torch
import pandas as pd
from PIL import Image

# Ensure app.py functions can be imported
sys.path.append(os.path.abspath("."))
%cd ..

# Import the core PyTorch editing function directly from app.py
from app import run_edit 

print("CUDA Available:", torch.cuda.is_available())
print("Active Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

/tmp/eoikonom/Thesis/FlowSlider


/tmp/eoikonom/Thesis/venv/lib64/python3.9/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


ModuleNotFoundError: No module named 'gradio'

In [ ]:
CSV_PATH = "../SliderEdit/experiments/experiments24-7.csv"
SOURCE_IMG_DIR = "../datasets/test_images"
OUTPUT_DIR = "../outputs/flowslider/1d"
STEERING_STEPS = [0.0, 0.5, 1.0, 2.0, 3.0, 4.0]

df_experiments = pd.read_csv(CSV_PATH)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Starting Direct PyTorch FlowSlider Benchmark...")

for idx, row in df_experiments.iterrows():
    sweep_id = str(row['id'])
    base_prompt = str(row['base_prompt'])
    subprompt1 = str(row['subprompt_1'])
    seed = int(row['seed'])

    exp_dir = os.path.join(OUTPUT_DIR, sweep_id)
    os.makedirs(exp_dir, exist_ok=True)

    source_img_path = os.path.join(SOURCE_IMG_DIR, f"{sweep_id}.png")
    if not os.path.exists(source_img_path):
        print(f"⚠️ [SKIP] Missing image at '{source_img_path}'")
        continue

    scales_str = ", ".join(map(str, STEERING_STEPS))
    clean_prompt = "".join(c for c in subprompt1 if c.isalnum() or c in (" ", "_")).replace(" ", "_")

    print(f"[{idx+1}/{len(df_experiments)}] Processing '{sweep_id}' locally on GPU...")

    try:
        # Call the PyTorch model logic directly—NO localhost, NO web server!
        results = run_edit(
            model_name="FLUX.1-dev",     # FLUX backbone model
            image=source_img_path,       # Source image path
            src_prompt=base_prompt,      # Source prompt
            tar_prompt=subprompt1,       # Target prompt
            tar_prompt_neg="",           # Negative prompt
            strengths_str=scales_str,    # Steering scales
            t_steps=28,                  # Diffusion timesteps
            n_max=20,                    # Flow editing steps
            src_cfg=3.5,                 # Source guidance
            tar_cfg=3.5,                 # Target guidance
            seed=float(seed)             # Random seed
        )

        # Handle returned images
        gallery_images = results if isinstance(results, list) else [results]
        
        for i, img_entry in enumerate(gallery_images):
            res_path = img_entry.get('image', {}).get('path') if isinstance(img_entry, dict) else img_entry
            if res_path and os.path.exists(res_path):
                out_img = Image.open(res_path).convert("RGB")
                s_val = STEERING_STEPS[i] if i < len(STEERING_STEPS) else i
                s_str = f"{s_val}".replace(".", "_")
                save_path = os.path.join(exp_dir, f"{sweep_id}_s_{s_str}_{clean_prompt}.png")
                out_img.save(save_path)

        print(f"  -> Finished {sweep_id}")

    except Exception as e:
        print(f"❌ Execution error for {sweep_id}: {e}")

print("Benchmark Complete!")

### HuggingFace Login

In [ ]:
from huggingface_hub import login

token_path = "/cs/student/msc/ml/2025/eoikonom/.hf_token"
if os.path.exists(token_path):
    with open(token_path, "r") as f:
        login(token=f.read().strip())
    print("Logged in via cluster token file.")
elif "HF_TOKEN" in os.environ:
    login(token=os.environ["HF_TOKEN"])
    print("Logged in via HF_TOKEN environment variable.")
else:
    print("Assuming local cached session.")

### Load CSV

In [ ]:
CSV_PATH = "./experiments24-7.csv"

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"Cannot find {CSV_PATH}.")

df_experiments = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df_experiments)} test cases.")

### Run FlowSlider

In [ ]:
def prepare_image_payload(file_path):
    try:
        f = handle_file(file_path)
        return f if isinstance(f, dict) else {"path": file_path}
    except Exception:
        return {"path": file_path}

def run_flowslider_remote_benchmark(
    df,
    client,
    steering_steps=[0, 0.5, 1.0, 2.0, 3.0, 4.0],
    output_root="./outputs/flowslider/1d",
    source_img_dir="./datasets/test_images"
):
    source_img_dir_abs = os.path.abspath(source_img_dir)
    output_root_abs = os.path.abspath(output_root)

    os.makedirs(source_img_dir_abs, exist_ok=True)
    os.makedirs(output_root_abs, exist_ok=True)

    print("==================================================")
    print("  STARTING REMOTE HF FLOWSLIDER BENCHMARK")
    print(f"  Source Images Directory: {source_img_dir_abs}")
    print(f"  Output Directory: {output_root_abs}")
    print(f"  Steering Scales (s): {steering_steps}")
    print("==================================================\n")

    for idx, row in df.iterrows():
        sweep_id = str(row['id'])
        subprompt1 = str(row['subprompt_1'])
        seed = int(row['seed'])
        base_prompt = str(row['base_prompt'])
        test_focus = str(row['test_focus'])

        exp_dir = os.path.join(output_root_abs, sweep_id)
        os.makedirs(exp_dir, exist_ok=True)

        source_img_path = os.path.join(source_img_dir_abs, f"{sweep_id}.png")

        if not os.path.exists(source_img_path):
            print(f"⚠️ [SKIP] Missing base image at '{source_img_path}'. Skipping row {idx}...")
            continue

        meta_data = {
            "id": sweep_id,
            "domain": row['domain'],
            "paradigm": "flowslider_hf_api",
            "base_prompt": base_prompt,
            "subprompt_1": subprompt1,
            "seed": seed,
            "steering_steps": steering_steps,
            "test_focus": test_focus
        }
        with open(os.path.join(exp_dir, "meta.json"), "w") as f:
            json.dump(meta_data, f, indent=4)

        clean_prompt = "".join(c for c in subprompt1 if c.isalnum() or c in (" ", "_")).replace(" ", "_")
        print(f"\n[{idx+1}/{len(df)}] Querying HF Space for '{sweep_id}': '{subprompt1}'")

        scales_str = ", ".join(map(str, steering_steps))

        try:
            # 💡 STRICT POSITIONAL ARGUMENTS MATCHING GRADIO API INDEX
            result = client.predict(
                "FLUX.1-dev",                           # 1. model_name
                prepare_image_payload(source_img_path), # 2. image
                base_prompt,                            # 3. src_prompt
                subprompt1,                             # 4. tar_prompt
                "",                                     # 5. tar_prompt_neg
                scales_str,                             # 6. strengths_str
                28,                                     # 7. t_steps
                20,                                     # 8. n_max
                3.5,                                    # 9. src_cfg
                3.5,                                    # 10. tar_cfg
                float(seed),                            # 11. seed
                api_name="/run_edit"
            )

            # Unpack returned gallery images from HF Space
            gallery_images = result if isinstance(result, list) else [result]
            outputs = []

            for i, img_entry in enumerate(gallery_images):
                res_path = None
                if isinstance(img_entry, dict):
                    img_data = img_entry.get('image', img_entry)
                    if isinstance(img_data, dict):
                        res_path = img_data.get('path') or img_data.get('url')
                    elif isinstance(img_data, str):
                        res_path = img_data
                elif isinstance(img_entry, str):
                    res_path = img_entry

                if res_path and os.path.exists(res_path):
                    out_img = Image.open(res_path).convert("RGB")
                    
                    s_val = steering_steps[i] if i < len(steering_steps) else i
                    s_str = f"{s_val}".replace(".", "_")
                    fname = f"{sweep_id}_s_{s_str}_{clean_prompt}.png"
                    full_save_path = os.path.join(exp_dir, fname)
                    
                    out_img.save(full_save_path)
                    outputs.append(out_img)
                    print(f"  -> Saved: {full_save_path}")

            if outputs:
                grid = make_image_grid([x.resize((128, 128)) for x in outputs], rows=1, cols=len(outputs))
                display(grid)

        except Exception as e:
            print(f"❌ Error during HF API call for {sweep_id}: {e}")

    print("\nFlowSlider Remote Benchmark Run Complete!")

In [ ]:
STEERING_SCALES = [0, 0.5, 1.0, 2.0, 3.0, 4.0]  # s=1 (standard edit), s>1 (amplified intensity) (s=0 for no edit????)

run_flowslider_remote_benchmark(
    df=df_experiments,
    client=client,
    steering_steps=STEERING_SCALES,
    source_img_dir="./datasets/test_images",
    output_root="./outputs/flowslider/1d"
)